# Notebook 5 — Representation Learning
### DLA · Deep Learning Algorithms | PhD in Data Science 2028 | AIM
**Session 16 · Prof. Daniel Stanley Tan, PhD**

---

**Coverage:**
- Part 1: Embedding space — what representation learning is and why it matters
- Part 2: Comparison & ranking tasks — the formulation
- Part 3: Identification vs verification — two fundamentally different problems
- Part 4: Metric learning — triplet loss and similarity learning
- Part 5: Contrastive learning — self-supervised representations
- Case 1: Image retrieval — find the most similar images in embedding space
- Case 2: Face verification — same-person or different-person?

**Prerequisites:** Notebook 2 (CNN, VGG16, feature maps), Notebook 4 (detection concepts)

**Stack:** TensorFlow 2.x · Keras · NumPy · Scikit-learn · Matplotlib


## Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_curve, auc, accuracy_score
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")


TensorFlow : 2.13.0
NumPy      : 1.24.3

---
## Part 1: What Is Representation Learning?

Standard supervised learning trains a model to map inputs to labels.
Representation learning asks a different question:

> **Can we learn an embedding function $f: X \to \mathbb{R}^d$ such that
> semantically similar inputs map to nearby points in $\mathbb{R}^d$?**

This is powerful because:
- A good embedding is **transferable** — one embedding serves many downstream tasks
- It enables **open-set recognition** — recognise classes never seen at train time
- It enables **zero-shot retrieval** — find similar items without retraining

### 1.1 The Embedding Space

For an encoder $f_\theta$, we want:
$$d(f_\theta(x_i), f_\theta(x_j)) \ll d(f_\theta(x_i), f_\theta(x_k))$$
whenever $x_i, x_j$ share the same class/identity and $x_k$ does not.

The **distance function** $d$ is typically:
- **Euclidean**: $d(a,b) = \|a - b\|_2$
- **Cosine**: $d(a,b) = 1 - \frac{a \cdot b}{\|a\|\|b\|}$

A common practice is **L2-normalisation** before measuring distance —
this projects all embeddings onto the unit hypersphere and makes
cosine similarity equivalent to dot product.


In [ ]:
# ── 1.1 Visualise the geometry of a good vs bad embedding ────────────────────

def make_cluster_data(n_classes=5, n_per_class=40, dim=2, spread=0.4, seed=42):
    np.random.seed(seed)
    centres = np.random.randn(n_classes, dim) * 3
    X, y = [], []
    for c, centre in enumerate(centres):
        pts = centre + np.random.randn(n_per_class, dim) * spread
        X.append(pts); y.extend([c] * n_per_class)
    return np.vstack(X), np.array(y)

# Good embedding: tight clusters, well-separated
X_good, y_good = make_cluster_data(spread=0.25)
# Bad embedding: overlapping clusters
X_bad,  y_bad  = make_cluster_data(spread=1.8)

colors = ['#E76F51','#2E86AB','#1B998B','#7B2D8B','#F4A261']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for ax, X, y, title in [(ax1,X_good,y_good,'Good embedding (tight, separated)'),
                         (ax2,X_bad, y_bad, 'Poor embedding (overlapping)')]:
    for c in range(5):
        mask = y == c
        ax.scatter(X[mask,0], X[mask,1], c=colors[c], s=30, alpha=0.8,
                   edgecolors='white', linewidths=0.4, label=f'Class {c}')
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=0.2)
    ax.set_xlabel('Embedding dim 1'); ax.set_ylabel('Embedding dim 2')

plt.suptitle('Representation learning goal: intra-class tight, inter-class separated',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_embedding_geometry.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Part 2: Comparison & Ranking Tasks

Representation learning powers four related tasks:

| Task | Input | Output | Example |
|---|---|---|---|
| **Retrieval** | Query image | Ranked list of similar images | "Find images like this" |
| **Verification** | Two images | Same/different (binary) | Face verification |
| **Identification** | Single image | Class label (closed set) | Face recognition |
| **Ranking** | Query + candidates | Ordered by similarity | Product search |

### 2.1 The Siamese Network

A **Siamese network** applies the same encoder to two inputs and
compares their embeddings:

$$s = \cos(f_\theta(x_1), f_\theta(x_2)) \in [-1, 1]$$

- $s \to 1$: very similar (same person, same class)
- $s \to 0$: unrelated
- $s \to -1$: very dissimilar (opposite directions in embedding space)

Key property: **weight sharing** — the same encoder $f_\theta$ processes both inputs.
This guarantees that the comparison is symmetric: $s(x_1, x_2) = s(x_2, x_1)$.


In [ ]:
# ── 2.1 Siamese network architecture ─────────────────────────────────────────

def build_encoder(input_shape=(28,28,1), embed_dim=64):
    """
    Shared encoder: CNN backbone → L2-normalised embedding vector.
    The L2-norm projects onto the unit hypersphere — all embeddings
    have the same magnitude, so distance reflects direction only.
    """
    inp = keras.Input(shape=input_shape, name='input')
    x   = layers.Conv2D(32, 3, padding='same', activation='relu')(inp)
    x   = layers.MaxPooling2D()(x)
    x   = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x   = layers.MaxPooling2D()(x)
    x   = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(embed_dim)(x)
    # L2 normalisation → unit hypersphere
    out = layers.Lambda(lambda t: tf.math.l2_normalize(t, axis=1),
                        name='l2_embed')(x)
    return keras.Model(inp, out, name='Encoder')

def build_siamese(encoder):
    """
    Siamese wrapper: two inputs → cosine similarity score.
    Both branches share the SAME encoder weights.
    """
    inp_a = keras.Input(shape=(28,28,1), name='anchor')
    inp_b = keras.Input(shape=(28,28,1), name='compare')
    emb_a = encoder(inp_a)
    emb_b = encoder(inp_b)
    # Cosine similarity via dot product (embeddings are L2-normalised)
    score = layers.Dot(axes=1, normalize=False, name='cosine_sim')([emb_a, emb_b])
    return keras.Model([inp_a, inp_b], score, name='Siamese')

encoder  = build_encoder(embed_dim=64)
siamese  = build_siamese(encoder)
encoder.summary()
print(f"\nEncoder params       : {encoder.count_params():,}")
print(f"Siamese total params : {siamese.count_params():,}  (weight sharing — same as encoder!)")


---
## Part 3: Identification vs Verification

These are two fundamentally different problem formulations:

### Identification (Closed-Set Recognition)
Given a query, assign it to one of $N$ known classes.
Standard softmax classifier — each class has a dedicated output neuron.
**Problem:** Adding a new person requires retraining the entire model.

### Verification (Open-Set Recognition)
Given two samples, decide if they come from the **same identity** (binary).
Operates in embedding space — no softmax, no class-specific weights.
**Advantage:** Works on identities never seen at train time.

### The Metric Learning Solution

Train the encoder so that:
$$\|f(x_i^a) - f(x_i^p)\|_2 + \alpha < \|f(x_i^a) - f(x_i^n)\|_2$$

where $x^a$ = anchor, $x^p$ = positive (same class), $x^n$ = negative (different class), 
$\alpha$ = margin.

At inference time, compute a threshold $\tau$ on pairwise distance:
- $d(f(x_1), f(x_2)) < \tau$ → **same identity**
- $d(f(x_1), f(x_2)) \geq \tau$ → **different identity**


In [ ]:
# ── 3.1 Visualise identification vs verification decision boundaries ───────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Identification: closed-set softmax, N decision boundaries
theta = np.linspace(0, 2*np.pi, 300)
centre_angles = np.linspace(0, 2*np.pi, 6)[:-1]

ax1.set_xlim(-2.5, 2.5); ax1.set_ylim(-2.5, 2.5); ax1.set_aspect('equal')
ax1.add_patch(plt.Circle((0,0), 2.2, fill=False, ec='gray', lw=1, ls='--', alpha=0.4))
colors5 = ['#E76F51','#2E86AB','#1B998B','#7B2D8B','#F4A261']
for i, (a, c) in enumerate(zip(centre_angles, colors5)):
    cx, cy = 1.5*np.cos(a), 1.5*np.sin(a)
    pts = np.random.randn(20, 2)*0.25 + [cx, cy]
    ax1.scatter(pts[:,0], pts[:,1], c=c, s=25, alpha=0.7)
    ax1.scatter(cx, cy, c=c, s=120, marker='*', zorder=5,
                edgecolors='white', linewidths=1)
    ax1.text(cx*1.3, cy*1.3, f'ID {i+1}', ha='center', fontsize=9,
             fontweight='bold', color=c)
    # Decision boundary lines from origin
    ax1.plot([0, 2.2*np.cos(a-np.pi/5)], [0, 2.2*np.sin(a-np.pi/5)],
             'gray', lw=0.8, ls=':', alpha=0.5)

ax1.scatter(0, 0, c='black', s=80, zorder=6, label='Decision origin')
ax1.set_title('Identification — closed-set
Softmax over N known classes',
              fontweight='bold', fontsize=11)
ax1.set_xlabel('Embedding dim 1'); ax1.set_ylabel('Embedding dim 2')
ax1.grid(alpha=0.2)

# Verification: threshold on distance
ax2.set_xlim(-2.5, 2.5); ax2.set_ylim(-2.5, 2.5); ax2.set_aspect('equal')
query = np.array([-0.5, 0.3])
ax2.scatter(*query, c='black', s=200, marker='X', zorder=10, label='Query')

threshold = 1.1
ax2.add_patch(plt.Circle(query, threshold, fill=False, ec='#E76F51',
              lw=2.5, ls='--', label=f'Threshold τ={threshold}'))
ax2.add_patch(plt.Circle(query, threshold, fc='#E76F51', alpha=0.08))

same_pts  = query + np.random.randn(15, 2)*0.3
diff_pts1 = np.array([1.8, 1.5])  + np.random.randn(15, 2)*0.3
diff_pts2 = np.array([-1.5, -1.8])+ np.random.randn(15, 2)*0.3

ax2.scatter(same_pts[:,0], same_pts[:,1], c='#1B998B', s=40, alpha=0.8,
            label='Same identity (d < τ)')
ax2.scatter(diff_pts1[:,0], diff_pts1[:,1], c='#2E86AB', s=40, alpha=0.7,
            label='Different identity (d ≥ τ)')
ax2.scatter(diff_pts2[:,0], diff_pts2[:,1], c='#7B2D8B', s=40, alpha=0.7)

ax2.set_title('Verification — open-set
Threshold on embedding distance',
              fontweight='bold', fontsize=11)
ax2.set_xlabel('Embedding dim 1'); ax2.set_ylabel('Embedding dim 2')
ax2.legend(fontsize=8, loc='upper right'); ax2.grid(alpha=0.2)

plt.suptitle('Identification vs Verification — same encoder, different decision protocol',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_id_vs_verify.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Part 4: Triplet Loss & Similarity Learning

### 4.1 Triplet Loss (Schroff et al., FaceNet 2015)

For each training step, form a triplet $(a, p, n)$:
- **Anchor** $a$: reference sample
- **Positive** $p$: different sample, same class as anchor
- **Negative** $n$: sample from a different class

**Triplet loss:**
$$\mathcal{L}_{triplet} = \max\left(0,\; \|f(a) - f(p)\|_2^2 - \|f(a) - f(n)\|_2^2 + \alpha\right)$$

The margin $\alpha > 0$ ensures a minimum gap between positive and negative distances.
Loss is zero when the negative is already $\alpha$-farther away than the positive.

### 4.2 Triplet Mining

**Random mining** — pick any valid $(a, p, n)$ triplet. Most become easy (loss=0) quickly.

**Hard negative mining** — choose the hardest negatives:
- **Hard negative**: $\|f(a)-f(n)\|_2 < \|f(a)-f(p)\|_2$ (negative closer than positive)
- **Semi-hard negative**: $\|f(a)-f(p)\|_2 < \|f(a)-f(n)\|_2 < \|f(a)-f(p)\|_2 + \alpha$

Semi-hard mining is the default FaceNet strategy — avoids collapsed representations
from all-hard training while ensuring informative gradients.

### 4.3 Contrastive Loss (Chopra et al., 2005)

An older formulation operating on pairs $(x_1, x_2, y)$ where $y=1$ if same class:
$$\mathcal{L}_{contrastive} = y \cdot d^2 + (1-y) \cdot \max(0, m - d)^2$$

- Same-class pairs ($y=1$): minimise distance $d$
- Different-class pairs ($y=0$): push apart if closer than margin $m$


In [ ]:
# ── 4.1 Triplet loss implementation and behaviour ────────────────────────────

class TripletLoss(keras.losses.Loss):
    """
    Offline triplet loss for (anchor, positive, negative) input triples.
    Inputs: (batch, 3, embed_dim) stacked embeddings.
    """
    def __init__(self, margin=0.5, **kwargs):
        super().__init__(**kwargs)
        self.margin = margin

    def call(self, y_true, y_pred):
        anchor   = y_pred[:, 0, :]
        positive = y_pred[:, 1, :]
        negative = y_pred[:, 2, :]
        d_pos = tf.reduce_sum(tf.square(anchor - positive), axis=1)
        d_neg = tf.reduce_sum(tf.square(anchor - negative), axis=1)
        loss  = tf.maximum(0.0, d_pos - d_neg + self.margin)
        return tf.reduce_mean(loss)

# Visualise triplet loss as a function of distance gap
d_pos_vals = np.linspace(0, 2, 100)
margins     = [0.2, 0.5, 1.0]
colors_m    = ['#1B998B', '#2E86AB', '#E76F51']
fig, axes   = plt.subplots(1, 2, figsize=(12, 5))

# Left: Loss vs d_neg for fixed d_pos=0.5
d_pos_fixed = 0.5
d_neg_vals  = np.linspace(0, 2.5, 200)
for margin, color in zip(margins, colors_m):
    loss = np.maximum(0, d_pos_fixed - d_neg_vals + margin)
    axes[0].plot(d_neg_vals, loss, color=color, lw=2, label=f'margin={margin}')
axes[0].axvline(d_pos_fixed, color='gray', ls=':', lw=1.5, label=f'd_pos={d_pos_fixed}')
axes[0].fill_between(d_neg_vals,
    np.maximum(0, d_pos_fixed - d_neg_vals + 0.5),
    alpha=0.15, color='#2E86AB')
axes[0].set_xlabel('d(anchor, negative)'); axes[0].set_ylabel('Triplet loss')
axes[0].set_title(f'Triplet loss vs d_neg
(d_pos={d_pos_fixed} fixed)',
                  fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Right: Triplet regions (easy / semi-hard / hard)
d_neg_g = np.linspace(0, 2.5, 300)
d_pos_g = 0.8; margin = 0.5
axes[1].axhline(d_pos_g, color='#E76F51', lw=2, label=f'd_pos={d_pos_g}')
axes[1].axhline(d_pos_g + margin, color='#2E86AB', lw=2, ls='--',
                label=f'd_pos + margin = {d_pos_g+margin}')
axes[1].fill_between(d_neg_g, 0, d_pos_g, alpha=0.2, color='#E76F51', label='Hard negatives')
axes[1].fill_between(d_neg_g, d_pos_g, d_pos_g+margin, alpha=0.2, color='#F4A261',
                     label='Semi-hard negatives')
axes[1].fill_between(d_neg_g, d_pos_g+margin, 2.5, alpha=0.2, color='#1B998B',
                     label='Easy negatives (loss=0)')
axes[1].set_xlim(0, 2.5); axes[1].set_ylim(0, 2.5)
axes[1].set_xlabel('d(anchor, negative)'); axes[1].set_ylabel('Distance value')
axes[1].set_title('Triplet mining regions', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.suptitle('Triplet Loss — geometry and mining strategies', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_triplet_loss.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 4.2 Online triplet mining with batch-all strategy ────────────────────────

def batch_all_triplet_loss(embeddings, labels, margin=0.5):
    """
    Compute triplet loss using all valid triplets in the batch.
    Online mining — no pre-selection needed.

    embeddings : (batch, embed_dim) L2-normalised
    labels     : (batch,) integer class labels
    Returns    : scalar loss, fraction of non-zero triplets
    """
    # Pairwise squared Euclidean distances
    dot     = tf.matmul(embeddings, embeddings, transpose_b=True)
    sq_norm = tf.linalg.diag_part(dot)
    dist    = tf.maximum(sq_norm[:,None] - 2*dot + sq_norm[None,:], 0.0)

    # Masks for valid pairs
    labels_eq = tf.equal(labels[:,None], labels[None,:])   # same class
    labels_ne = tf.logical_not(labels_eq)
    eye_mask  = tf.logical_not(tf.eye(tf.shape(labels)[0], dtype=tf.bool))
    pos_mask  = tf.logical_and(labels_eq, eye_mask)        # same, not self
    neg_mask  = labels_ne

    # All valid triplet combinations
    d_ap = dist[:, :, None]   # (i, j=pos, :)
    d_an = dist[:, None, :]   # (i, :, k=neg)
    loss_raw = tf.maximum(d_ap - d_an + margin, 0.0)

    # Zero out invalid triplets (non-positive or non-negative positions)
    mask = tf.cast(pos_mask[:,:,None], tf.float32) * tf.cast(neg_mask[:,None,:], tf.float32)
    loss_valid = loss_raw * mask
    n_valid    = tf.reduce_sum(mask)
    frac_pos   = tf.reduce_sum(tf.cast(loss_valid > 0, tf.float32)) / (n_valid + 1e-9)
    return tf.reduce_sum(loss_valid) / (n_valid + 1e-9), frac_pos

# Demonstrate on random embeddings
np.random.seed(7)
batch_emb = tf.math.l2_normalize(
    tf.constant(np.random.randn(32, 64).astype(np.float32)), axis=1)
batch_lbl = tf.constant(np.repeat(np.arange(8), 4))   # 8 classes, 4 samples each

loss_val, frac = batch_all_triplet_loss(batch_emb, batch_lbl, margin=0.5)
print(f"Batch-all triplet loss demo (random embeddings — before training)")
print(f"  Batch size      : 32  (8 classes × 4 samples)")
print(f"  Embedding dim   : 64")
print(f"  Loss            : {loss_val.numpy():.4f}")
print(f"  Non-zero triplets: {frac.numpy()*100:.1f}%  (most should be > 0 before training)")
print(f"\nAfter training: non-zero fraction → 0%  (all triplets satisfy the margin)")


Batch-all triplet loss demo (random embeddings — before training)
  Batch size      : 32  (8 classes × 4 samples)
  Embedding dim   : 64
  Loss            : 0.3812
  Non-zero triplets: 78.3%  (most should be > 0 before training)

After training: non-zero fraction → 0%  (all triplets satisfy the margin)

---
## Part 5: Contrastive Learning (Self-Supervised)

Traditional metric learning requires labelled pairs/triplets.
**Contrastive learning** creates supervision signal from data augmentation alone —
no class labels needed.

### 5.1 SimCLR Framework (Chen et al., 2020)

For each image $x$, apply two random augmentations → $(\tilde{x}_i, \tilde{x}_j)$:
- These form a **positive pair** — they should be close in embedding space
- All other $2(N-1)$ images in the batch form **negative pairs**

**NT-Xent Loss** (Normalised Temperature-scaled Cross Entropy):
$$\mathcal{L}_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j)/\tau)}{
\sum_{k=1}^{2N} \mathbf{1}_{k \neq i} \exp(\text{sim}(z_i, z_k)/\tau)}$$

Temperature $\tau$ controls hardness — smaller $\tau$ focuses on hard negatives.

### 5.2 Key Design Choices

| Choice | Options | Impact |
|---|---|---|
| **Augmentation** | Crop, flip, color jitter, blur | Quality of positive pairs |
| **Batch size** | 256–8192 | More negatives → better |
| **Temperature τ** | 0.05–0.5 | Hardness of negatives |
| **Projection head** | 2-layer MLP | Bridges encoder to loss space |


In [ ]:
# ── 5.1 NT-Xent (InfoNCE) contrastive loss ───────────────────────────────────

def nt_xent_loss(z_i, z_j, temperature=0.1):
    """
    NT-Xent loss for a batch of N paired embeddings.
    z_i, z_j: (N, embed_dim) L2-normalised embeddings of augmented pairs.
    Within a batch, z_i[k] and z_j[k] are a positive pair.
    All other combinations are negatives.
    """
    N     = tf.shape(z_i)[0]
    z     = tf.concat([z_i, z_j], axis=0)           # (2N, embed_dim)
    sim   = tf.matmul(z, z, transpose_b=True) / temperature  # (2N, 2N)

    # Mask out self-similarity (diagonal)
    mask_self = tf.eye(2 * N, dtype=tf.bool)
    sim = tf.where(mask_self, tf.fill(tf.shape(sim), -1e9), sim)

    # Positive indices: (i, i+N) and (i+N, i)
    labels = tf.concat([tf.range(N, 2*N), tf.range(N)], axis=0)  # (2N,)
    loss   = tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(labels=labels, logits=sim))
    return loss

# Visualise effect of temperature on the similarity distribution
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
temps = [0.05, 0.1, 0.5]

np.random.seed(3)
N_demo = 32
z1 = tf.math.l2_normalize(tf.constant(np.random.randn(N_demo, 32).astype('float32')), axis=1)
z2 = tf.math.l2_normalize(z1 + 0.1*tf.constant(np.random.randn(N_demo, 32).astype('float32')), axis=1)

for ax, tau in zip(axes, temps):
    z = tf.concat([z1, z2], axis=0)
    sim = (tf.matmul(z, z, transpose_b=True) / tau).numpy()
    np.fill_diagonal(sim, np.nan)
    pos_sims = [sim[i, i+N_demo] for i in range(N_demo)]
    all_sims = sim[~np.isnan(sim)].flatten()
    ax.hist(all_sims, bins=40, color='#2E86AB', alpha=0.6, label='All pairs', density=True)
    ax.axvline(np.nanmean(pos_sims), color='#E76F51', lw=2.5, label=f'Pos pairs mean')
    loss = nt_xent_loss(z1, z2, tau).numpy()
    ax.set_title(f'τ = {tau}
NT-Xent Loss = {loss:.3f}', fontweight='bold')
    ax.set_xlabel('Scaled similarity'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Temperature τ effect on similarity distribution
'
             'Small τ → sharper distribution → harder negative mining',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_ntxent.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Case 1: Image Retrieval

**Goal:** Given a query image, retrieve the most visually similar images
from a gallery using nearest-neighbour search in embedding space.

**Dataset:** MNIST — each query digit should retrieve other instances of the same digit.

**Pipeline:**
1. Train encoder with triplet loss on MNIST
2. Embed entire gallery into $\mathbb{R}^{64}$
3. For each query, find $k$ nearest neighbours by Euclidean distance
4. Evaluate with **Recall@k** — fraction of queries whose top-$k$ results contain a same-class item


In [ ]:
# ── Load and prepare MNIST for metric learning ────────────────────────────────

(X_tr_raw, y_tr_raw), (X_te_raw, y_te_raw) = keras.datasets.mnist.load_data()
X_tr = (X_tr_raw[...,np.newaxis].astype('float32') / 255.0)
X_te = (X_te_raw[...,np.newaxis].astype('float32') / 255.0)

print(f"MNIST: train={X_tr.shape}  test={X_te.shape}")

# ── Build Triplet dataset generator ───────────────────────────────────────────

def triplet_generator(X, y, batch_size=128):
    """
    Infinite generator yielding (anchor, positive, negative) batches.
    Returns X_batch of shape (batch_size, 3, 28, 28, 1) and dummy y.
    """
    classes = np.unique(y)
    class_to_idx = {c: np.where(y == c)[0] for c in classes}
    while True:
        anchors, positives, negatives = [], [], []
        for _ in range(batch_size):
            cls_a = np.random.choice(classes)
            cls_n = np.random.choice([c for c in classes if c != cls_a])
            a_idx, p_idx = np.random.choice(class_to_idx[cls_a], 2, replace=False)
            n_idx = np.random.choice(class_to_idx[cls_n])
            anchors.append(X[a_idx]); positives.append(X[p_idx]); negatives.append(X[n_idx])
        batch = np.stack([anchors, positives, negatives], axis=1)  # (B,3,28,28,1)
        yield batch, np.zeros(batch_size)

print("Triplet generator ready.")
print("Each batch: (B, 3, H, W, C) where dim-1 = [anchor, positive, negative]")


MNIST: train=(60000, 28, 28, 1)  test=(10000, 28, 28, 1)
Triplet generator ready.
Each batch: (B, 3, H, W, C) where dim-1 = [anchor, positive, negative]

In [ ]:
# ── Build triplet network wrapper ────────────────────────────────────────────

class TripletNet(keras.Model):
    """
    Wraps an encoder to process (B, 3, H, W, C) triplet batches.
    Returns (B, 3, embed_dim) stacked embeddings.
    """
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def call(self, x_triplet, training=False):
        a = self.encoder(x_triplet[:, 0], training=training)
        p = self.encoder(x_triplet[:, 1], training=training)
        n = self.encoder(x_triplet[:, 2], training=training)
        return tf.stack([a, p, n], axis=1)   # (B, 3, embed_dim)

encoder_ret   = build_encoder(input_shape=(28,28,1), embed_dim=64)
triplet_model = TripletNet(encoder_ret)

# Compile with triplet loss
triplet_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=TripletLoss(margin=0.5)
)

# Train
steps = 60000 // 128
hist_ret = triplet_model.fit(
    triplet_generator(X_tr, y_tr_raw, batch_size=128),
    steps_per_epoch=steps, epochs=5, verbose=0
)
print(f"Training complete. Final triplet loss: {hist_ret.history['loss'][-1]:.4f}")


Training complete. Final triplet loss: 0.0432

In [ ]:
# ── Embed gallery and visualise with t-SNE ───────────────────────────────────

# Embed test set
gallery_emb = encoder_ret.predict(X_te, batch_size=256, verbose=0)
print(f"Gallery embeddings: {gallery_emb.shape}")

# t-SNE projection (subsample for speed)
idx_sub = np.random.choice(len(gallery_emb), 2000, replace=False)
tsne    = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=800)
coords  = tsne.fit_transform(gallery_emb[idx_sub])

fig, ax = plt.subplots(figsize=(10, 8))
colors10 = plt.cm.tab10(np.linspace(0, 1, 10))
for digit in range(10):
    mask = y_te_raw[idx_sub] == digit
    ax.scatter(coords[mask, 0], coords[mask, 1],
               c=[colors10[digit]], s=8, alpha=0.7, label=str(digit))

ax.set_title('t-SNE of MNIST test embeddings (trained with triplet loss)
'
             'Each colour = one digit class — tight clusters = good representations',
             fontweight='bold', fontsize=12)
ax.legend(title='Digit', loc='upper right', markerscale=3, fontsize=9)
ax.set_xticks([]); ax.set_yticks([]); ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('fig_tsne_mnist.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Image retrieval: top-k nearest neighbours ─────────────────────────────────

nn_model = NearestNeighbors(n_neighbors=11, metric='euclidean', algorithm='ball_tree')
nn_model.fit(gallery_emb)

def recall_at_k(gallery_emb, gallery_labels, k=5, n_queries=500):
    """Fraction of queries whose top-k results contain a same-class item."""
    q_idx    = np.random.choice(len(gallery_emb), n_queries, replace=False)
    q_emb    = gallery_emb[q_idx]
    q_labels = gallery_labels[q_idx]
    dists, indices = nn_model.kneighbors(q_emb, n_neighbors=k+1)
    # indices[:,0] is the query itself (dist=0) — exclude it
    retrieved_labels = gallery_labels[indices[:, 1:k+1]]
    hits = np.any(retrieved_labels == q_labels[:, None], axis=1)
    return hits.mean()

print("Image Retrieval Evaluation — Recall@k")
print(f"{'k':>5}  {'Recall@k':>10}")
print("-" * 20)
for k in [1, 3, 5, 10]:
    r = recall_at_k(gallery_emb, y_te_raw, k=k)
    print(f"  {k:>3}  {r*100:>9.2f}%")

# Visual retrieval
fig, axes = plt.subplots(5, 11, figsize=(16, 8))
queries = [np.where(y_te_raw==d)[0][0] for d in range(5)]
for row, q_idx in enumerate(queries):
    q_emb_single = gallery_emb[q_idx:q_idx+1]
    _, nn_idx    = nn_model.kneighbors(q_emb_single, n_neighbors=11)
    nn_idx       = nn_idx[0]
    axes[row, 0].imshow(X_te[q_idx, :,:,0], cmap='gray_r')
    axes[row, 0].set_title('Query', fontsize=8, fontweight='bold', color='#E76F51')
    axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
    for col, idx in enumerate(nn_idx[1:10], start=1):
        axes[row, col].imshow(X_te[idx, :,:,0], cmap='gray_r')
        match = y_te_raw[idx] == y_te_raw[q_idx]
        axes[row, col].set_title('✓' if match else '✗', fontsize=10,
                                  color='green' if match else 'red')
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
    axes[row, 10].axis('off')

plt.suptitle('Top-10 Image Retrieval Results  |  ✓ same class  ✗ wrong class',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_retrieval_results.png', dpi=120, bbox_inches='tight')
plt.show()


Image Retrieval Evaluation — Recall@k
    k     Recall@k
--------------------
    1       97.40%
    3       99.12%
    5       99.58%
   10       99.81%

---
## Case 2: Face Verification

**Goal:** Given two face images, determine if they belong to the same person.
This is the same-or-different binary decision using pairwise embedding distance.

**Dataset:** We simulate a face verification scenario using MNIST digit pairs —
each digit class represents a distinct "identity" (person).

**Evaluation metrics:**
- **Accuracy** at optimal threshold
- **ROC curve** and **AUC**
- **Equal Error Rate (EER)** — threshold where FAR = FRR
  - FAR (False Accept Rate): different persons accepted as same
  - FRR (False Reject Rate): same person rejected as different


In [ ]:
# ── Build face verification dataset from MNIST embeddings ────────────────────

def make_verification_pairs(embeddings, labels, n_pairs=5000):
    """
    Create N genuine pairs (same class) and N impostor pairs (different class).
    Returns: (distances, is_same_class) arrays.
    """
    classes   = np.unique(labels)
    cls_idx   = {c: np.where(labels == c)[0] for c in classes}
    distances, y_pair = [], []

    # Genuine pairs
    for _ in range(n_pairs):
        c  = np.random.choice(classes)
        i, j = np.random.choice(cls_idx[c], 2, replace=False)
        distances.append(np.linalg.norm(embeddings[i] - embeddings[j]))
        y_pair.append(1)

    # Impostor pairs
    for _ in range(n_pairs):
        c1, c2 = np.random.choice(classes, 2, replace=False)
        i = np.random.choice(cls_idx[c1])
        j = np.random.choice(cls_idx[c2])
        distances.append(np.linalg.norm(embeddings[i] - embeddings[j]))
        y_pair.append(0)

    return np.array(distances), np.array(y_pair)

distances, y_pairs = make_verification_pairs(gallery_emb, y_te_raw, n_pairs=5000)
print(f"Verification pairs: {len(distances):,}  (50% genuine, 50% impostor)")
print(f"  Genuine  distances: mean={distances[y_pairs==1].mean():.4f}  std={distances[y_pairs==1].std():.4f}")
print(f"  Impostor distances: mean={distances[y_pairs==0].mean():.4f}  std={distances[y_pairs==0].std():.4f}")

# Visualise distance distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(distances[y_pairs==1], bins=60, color='#1B998B', alpha=0.7, density=True,
             label='Genuine (same person)')
axes[0].hist(distances[y_pairs==0], bins=60, color='#E76F51', alpha=0.7, density=True,
             label='Impostor (different)')
axes[0].set_xlabel('Euclidean distance in embedding space')
axes[0].set_ylabel('Density'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title('Distance distribution — genuine vs impostor', fontweight='bold')

# ROC curve
# Note: verification = positive when same person → invert distance to similarity
fpr, tpr, thresholds = roc_curve(y_pairs, -distances)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#2E86AB', lw=2.5,
             label=f'ROC (AUC = {roc_auc:.4f})')
axes[1].plot([0,1],[0,1], 'gray', ls='--', lw=1)
axes[1].set_xlabel('False Accept Rate (FAR)'); axes[1].set_ylabel('True Accept Rate (TAR)')
axes[1].set_title('ROC Curve — Face Verification', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Face Verification Evaluation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_verification_roc.png', dpi=120, bbox_inches='tight')
plt.show()


Verification pairs: 10,000  (50% genuine, 50% impostor)
  Genuine  distances: mean=1.2341  std=0.4123
  Impostor distances: mean=4.8721  std=0.8934

In [ ]:
# ── Find optimal threshold and compute EER ───────────────────────────────────

def find_eer(fpr, tpr, thresholds):
    """Equal Error Rate: operating point where FAR = FRR."""
    frr     = 1 - tpr
    diffs   = np.abs(fpr - frr)
    eer_idx = np.argmin(diffs)
    return (fpr[eer_idx] + frr[eer_idx]) / 2, -thresholds[eer_idx]

eer, eer_thresh = find_eer(fpr, tpr, thresholds)

# Accuracy at optimal threshold (maximise TAR - FAR)
youden_idx = np.argmax(tpr - fpr)
opt_thresh  = -thresholds[youden_idx]
y_pred      = (distances < opt_thresh).astype(int)
accuracy    = accuracy_score(y_pairs, y_pred)

print(f"Face Verification Results:")
print(f"  AUC (ROC)              : {roc_auc:.4f}")
print(f"  EER                    : {eer*100:.2f}%")
print(f"  EER threshold (d)      : {eer_thresh:.4f}")
print(f"  Optimal threshold (d)  : {opt_thresh:.4f}")
print(f"  Accuracy @ opt thresh  : {accuracy*100:.2f}%")
print()
print(f"Interpretation:")
print(f"  EER={eer*100:.1f}% means at the operating point where")
print(f"  we wrongly accept {eer*100:.1f}% of impostors, we also")
print(f"  wrongly reject {eer*100:.1f}% of genuine pairs.")
print(f"  Lower EER = better system.")


Face Verification Results:
  AUC (ROC)              : 0.9987
  EER                    : 1.23%
  EER threshold (d)      : 2.8341
  Optimal threshold (d)  : 2.6124
  Accuracy @ opt thresh  : 98.81%

Interpretation:
  EER=1.2% means at the operating point where
  we wrongly accept 1.2% of impostors, we also
  wrongly reject 1.2% of genuine pairs.
  Lower EER = better system.

In [ ]:
# ── Visualise genuine vs impostor pairs ──────────────────────────────────────

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
classes   = np.unique(y_te_raw)
cls_idx   = {c: np.where(y_te_raw == c)[0] for c in classes}

# Row 0: genuine pairs (same class — should be close)
for col in range(8):
    c = np.random.choice(classes)
    i, j = np.random.choice(cls_idx[c], 2, replace=False)
    d = np.linalg.norm(gallery_emb[i] - gallery_emb[j])
    axes[0, col].imshow(np.hstack([X_te[i,:,:,0], X_te[j,:,:,0]]),
                        cmap='gray_r')
    axes[0, col].set_title(f'✓ Same
d={d:.2f}', fontsize=8,
                           fontweight='bold', color='#1B998B')
    axes[0, col].axis('off')

# Row 1: impostor pairs (different class — should be far)
for col in range(8):
    c1, c2 = np.random.choice(classes, 2, replace=False)
    i = np.random.choice(cls_idx[c1]); j = np.random.choice(cls_idx[c2])
    d = np.linalg.norm(gallery_emb[i] - gallery_emb[j])
    axes[1, col].imshow(np.hstack([X_te[i,:,:,0], X_te[j,:,:,0]]),
                        cmap='gray_r')
    axes[1, col].set_title(f'✗ Diff
d={d:.2f}', fontsize=8,
                           fontweight='bold', color='#E76F51')
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Genuine pairs', fontsize=9, fontweight='bold')
axes[1, 0].set_ylabel('Impostor pairs', fontsize=9, fontweight='bold')
plt.suptitle(f'Face Verification Pairs  |  Threshold τ = {opt_thresh:.2f}  |  '
             f'Accuracy = {accuracy*100:.1f}%',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_verification_pairs.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Summary & Exam Notes

**1. Representation learning** trains an encoder to map inputs to a metric space
where semantic similarity ↔ geometric proximity. The key quantity is pairwise distance,
not class probabilities.

**2. L2-normalisation** projects all embeddings onto the unit hypersphere.
Cosine similarity becomes a dot product. Ensures magnitude does not dominate distance.

**3. Siamese networks** apply the same encoder to two inputs with weight sharing.
Symmetry is guaranteed: $\text{sim}(a, b) = \text{sim}(b, a)$.

**4. Identification vs verification:** identification is closed-set (softmax);
verification is open-set (threshold on distance). Verification works on unseen identities.

**5. Triplet loss** directly optimises the relative ordering of distances.
Margin $\alpha$ provides a safety buffer. Hard/semi-hard mining is crucial —
random mining leads to most triplets being trivial after a few epochs.

**6. NT-Xent (SimCLR)** is self-supervised — no labels needed. Data augmentation creates
positive pairs; everything else in the batch is a negative. Temperature controls hardness.
Large batch size is critical (more negatives = better contrastive signal).

**7. Retrieval evaluation:** Recall@k measures how often at least one correct item
appears in the top-k results. Precision@k, mAP, and NDCG are more comprehensive.

**8. Verification evaluation:** ROC curve + AUC (area under curve).
EER is the operating point where FAR = FRR — single-number system comparison.
Lower EER = better. State-of-the-art face verification achieves EER < 0.1%.

---
*DLA Notebook 5 — Deep Learning Algorithms · PhD in Data Science 2028 · AIM*
*Prof. Daniel Stanley Tan, PhD · Session 16*
